In [ ]:
# =========================
# 0) Imports & Config
# =========================
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union, Dict
import re, ast
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---- Base paths ----
BASE_DIR = Path("/Users/curiostudio/Desktop/iclr-target_neuron_ablation/data").expanduser().resolve()
OUT_DIR  = None  # None -> BASE_DIR / "plots_scheme_A_styled"
print("BASE_DIR =", BASE_DIR)

# ---- Matplotlib style (publication-friendly) ----
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.linestyle': '-',
    'grid.linewidth': 0.4,
})

# =========================
# 1) Discovery & Data Utils
# =========================
@dataclass(frozen=True)
class CsvEntry:
    elbow: int          # e.g., 5,10,...
    model: str          # e.g., "EleutherAI/pythia-1B-deduped"
    mode: str           # "boost" or "suppress"
    path: Path          # CSV file path

def discover_csvs(base_dir: Path) -> List[CsvEntry]:
    """
    Compatible with:
      A) base/elbow_10/.../prob/longtail/(boost|suppress)/500_all.csv
      B) base/0_10/...   /prob/longtail/(boost|suppress)/500_all.csv

    elbow is parsed from the TOP directory under base_dir by grabbing
    the last integer in that directory name (e.g., 'elbow_10'->10, '0_10'->10).
    model is path[1:prob_idx] joined by '/', mode is 'longtail' child.
    """
    base_dir = Path(base_dir).expanduser().resolve()
    entries: List[CsvEntry] = []

    # find all .../prob/longtail/<mode>/500_all.csv
    for csv_path in base_dir.rglob("prob/longtail/*/500_all.csv"):
        try:
            rel = csv_path.relative_to(base_dir)
        except ValueError:
            continue
        parts = rel.parts
        if len(parts) < 5:
            continue

        # top dir contains elbow token
        top_dir = parts[0]
        m = re.search(r'(\d+)(?!.*\d)', top_dir)
        if not m:
            continue
        elbow = int(m.group(1))

        # find 'prob' index for model slicing
        try:
            prob_idx = parts.index("prob")
        except ValueError:
            continue

        # model spans [1:prob_idx)
        model_parts = parts[1:prob_idx]
        model = "/".join(model_parts) if model_parts else "UNKNOWN_MODEL"

        # mode under 'longtail'
        try:
            lt_idx = parts.index("longtail")
            mode = parts[lt_idx + 1]
        except Exception:
            mode = csv_path.parent.name

        if mode not in {"boost", "suppress"}:
            continue

        entries.append(CsvEntry(elbow=elbow, model=model, mode=mode, path=csv_path.resolve()))

    entries.sort(key=lambda e: (e.model, e.mode, e.elbow))
    return entries

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def safe_name(s: str) -> str:
    return s.replace("/", "__").replace(" ", "_")

def _parse_series_of_list_strings(series: pd.Series) -> np.ndarray:
    """
    Parse a column where each cell is a stringified list, like "[0.1, 0.2, ...]".
    Concatenate into a flat float array. Robust to bad rows.
    """
    chunks = []
    for s in series:
        if isinstance(s, str):
            t = s.strip()
            if t.startswith("[") and t.endswith("]"):
                try:
                    arr = np.array(ast.literal_eval(t), dtype=float)
                    chunks.append(arr)
                except Exception:
                    continue
        elif isinstance(s, (list, tuple, np.ndarray)):
            chunks.append(np.array(s, dtype=float))
    if not chunks:
        return np.array([], dtype=float)
    return np.concatenate(chunks)

def load_values_for_step(
    csv_path: Path,
    step_choice: Union[int, str, None] = "max",     # "max" or an integer
    value_col: str = "abs_delta_loss_post_ablation",
    step_col: str = "step"
) -> np.ndarray:
    """
    Read csv, pick last step or a specific step. Parse stringified-list column,
    flatten to 1D array. Keep positive finite values only.
    """
    df = pd.read_csv(csv_path)
    if step_col not in df.columns or value_col not in df.columns:
        raise ValueError(f"{csv_path} missing '{step_col}' and/or '{value_col}'")
    step_target = df[step_col].max() if (step_choice in ("max", None)) else int(step_choice)
    sub = df.loc[df[step_col] == step_target, value_col]
    vals = _parse_series_of_list_strings(sub)
    vals = vals[np.isfinite(vals) & (vals > 0)]
    if vals.size == 0:
        raise ValueError(f"{csv_path} step={step_target}: no positive finite values in '{value_col}'")
    return vals

def zipf_transform(values: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Sort descending; return log(rank), log(value).
    """
    v = np.asarray(values, float)
    v = v[np.isfinite(v) & (v > 0)]
    if v.size == 0:
        raise ValueError("No positive finite values to analyze.")
    srt = -np.sort(-v)
    ranks = np.arange(1, srt.size + 1, 1, dtype=float)
    return np.log(ranks), np.log(srt)

def fit_linear_region(log_ranks: np.ndarray, log_vals: np.ndarray,
                      region: Optional[Tuple[float, float]]):
    """
    Fit y = a*x + b on [x1,x2] (if region provided) or globally.
    Return (a, b, R^2). Power-law exponent alpha = -a.
    """
    x, y = np.asarray(log_ranks), np.asarray(log_vals)
    if region is not None:
        lo, hi = region
        m = (x >= lo) & (x <= hi)
        x, y = x[m], y[m]
    if x.size < 3:
        return None
    a, b = np.polyfit(x, y, 1)
    yhat = a*x + b
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else float("nan")
    return (a, b, r2)

# ---- Discover once ----
entries: List[CsvEntry] = discover_csvs(BASE_DIR)
MODELS = sorted({e.model for e in entries})
MODES  = sorted({e.mode  for e in entries})
ELBOWS = sorted({e.elbow for e in entries})
print(f"Found CSVs: {len(entries)}")
print(f"Models ({len(MODELS)}): {MODELS}")
print(f"Modes: {MODES}")
print(f"Elbows: {ELBOWS}")

def family_of(model: str) -> str:
    m = model.lower()
    if "pythia" in m or "eleutherai" in m: return "Pythia"
    if "gpt2"   in m:                       return "GPT"
    return "Other"

def available_steps_for(model: str, mode: str, elbow: Optional[int] = None) -> List[int]:
    """
    Collect all steps present across CSVs for (model, mode),
    or for the single CSV of (model, mode, elbow) if elbow is provided.
    """
    steps = set()
    for e in entries:
        if e.model == model and e.mode == mode and (elbow is None or e.elbow == elbow):
            try:
                s = pd.read_csv(e.path, usecols=["step"])["step"].unique().tolist()
                steps.update(s)
            except Exception:
                pass
    return sorted(int(x) for x in steps)

# =========================
# 2) Plotting Functions
# =========================
def plot_model_across_elbows(
    model: str,
    mode: str,
    step_choice: Union[int, str] = "max",
    linear_region: Optional[Tuple[float, float]] = (2.0, 3.3),
    ylim: Optional[Tuple[float, float]] = None,
    dpi: int = 300,
    cmap_name: str = "plasma",
    extend_fit_to_full_span: bool = True,
    show_colorbar: bool = True,
    png_only: bool = True
):
    """
    Single model & mode. Overlay different elbows at a chosen step.
    Sequential colormap encodes elbow (small->large). English labels.
    """
    subset = [e for e in entries if e.model == model and e.mode == mode]
    subset = sorted(subset, key=lambda e: e.elbow)
    if not subset:
        print("No data matched for the given (model, mode).")
        return None

    elbows = np.array(sorted({e.elbow for e in subset}), dtype=float)
    norm = mpl.colors.Normalize(vmin=float(elbows.min()), vmax=float(elbows.max()))
    cmap = mpl.cm.get_cmap(cmap_name)
    color_of = lambda el: cmap(norm(float(el)))

    markers = ['o','s','^','D','v','P','X','*']
    fig, ax = plt.subplots(figsize=(9.6, 6.2), dpi=dpi)
    if show_colorbar:
        plt.subplots_adjust(right=0.86)

    global_x_min, global_x_max = np.inf, -np.inf
    cached, skipped = [], []

    for i, ent in enumerate(subset):
        try:
            vals = load_values_for_step(ent.path, step_choice=step_choice)
            xr, yr = zipf_transform(vals)
        except Exception as ex:
            skipped.append((ent.elbow, str(ex)))
            continue
        global_x_min = min(global_x_min, xr.min())
        global_x_max = max(global_x_max, xr.max())
        cached.append((ent.elbow, xr, yr, color_of(ent.elbow), markers[i % len(markers)]))

    for el, xr, yr, c, mk in cached:
        ax.scatter(xr, yr, s=12, alpha=0.45, c=[c], marker=mk,
                   edgecolors='white', linewidths=0.3, zorder=1)

    for el, xr, yr, c, mk in cached:
        fit = fit_linear_region(xr, yr, linear_region)
        if fit is None:
            continue
        a, b, r2 = fit
        x_fit = np.linspace(global_x_min, global_x_max, 256) if extend_fit_to_full_span \
                else (xr if linear_region is None else xr[(xr>=linear_region[0]) & (xr<=linear_region[1])])
        y_fit = a * x_fit + b
        ax.plot(x_fit, y_fit, linewidth=1.2, color=c, zorder=2)

    step_str = "max" if (step_choice == "max" or step_choice is None) else str(step_choice)
    ax.set_title(f"Zipf: model={model} | mode={mode} | step={step_str} | across elbows")
    ax.set_xlabel("log(rank)"); ax.set_ylabel("log(value)")
    if ylim is not None: ax.set_ylim(*ylim)

    if show_colorbar:
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("elbow")

    ax.margins(x=0.02, y=0.05); fig.tight_layout()

    out_root = (OUT_DIR if OUT_DIR is not None else BASE_DIR / "plots_scheme_A_styled")
    out_dir  = ensure_dir(out_root / "by_model_across_elbows")
    fname    = f"{safe_name(model)}_{mode}_step{step_str}_across_elbows_seq.png"
    fpath    = out_dir / fname
    fig.savefig(fpath, dpi=dpi, bbox_inches="tight")
    plt.show()

    if skipped:
        print("Skipped elbows:", skipped)
    return fpath

def plot_model_across_steps_at_elbow(
    model: str,
    mode: str,
    elbow: int,
    steps_to_compare: Optional[List[int]] = None,   # None -> all steps in this CSV
    linear_region: Optional[Tuple[float, float]] = (2.0, 3.3),
    ylim: Optional[Tuple[float, float]] = None,
    dpi: int = 300,
    cmap_name: str = "plasma",
    extend_fit_to_full_span: bool = True,
    show_colorbar: bool = True,
    png_only: bool = True
):
    """
    Fixed elbow. Overlay multiple steps for the same (model, mode).
    Sequential colormap encodes step (small->large). English labels.
    """
    grp = [e for e in entries if e.model == model and e.mode == mode and e.elbow == elbow]
    if not grp:
        print("No CSV found for the given (model, mode, elbow).")
        return None
    csv_path = grp[0].path

    try:
        avail_steps = sorted(pd.read_csv(csv_path, usecols=["step"])["step"].unique().tolist())
    except Exception as ex:
        print("Failed to read steps:", ex)
        return None

    steps = avail_steps if (steps_to_compare is None or len(steps_to_compare) == 0) \
            else [int(s) for s in steps_to_compare if int(s) in avail_steps]
    if not steps:
        print("No valid steps to compare.")
        return None

    steps_arr = np.array(steps, dtype=float)
    norm = mpl.colors.Normalize(vmin=float(steps_arr.min()), vmax=float(steps_arr.max()))
    cmap = mpl.cm.get_cmap(cmap_name)
    color_of = lambda st: cmap(norm(float(st)))

    markers = ['o','s','^','D','v','P','X','*']
    fig, ax = plt.subplots(figsize=(9.6, 6.2), dpi=dpi)
    if show_colorbar:
        plt.subplots_adjust(right=0.86)

    global_x_min, global_x_max = np.inf, -np.inf
    cached, skipped = [], []

    for i, st in enumerate(steps):
        try:
            vals = load_values_for_step(csv_path, step_choice=int(st))
            xr, yr = zipf_transform(vals)
        except Exception as ex:
            skipped.append((st, str(ex))); continue
        global_x_min = min(global_x_min, xr.min())
        global_x_max = max(global_x_max, xr.max())
        cached.append((st, xr, yr, color_of(st), markers[i % len(markers)]))

    for st, xr, yr, c, mk in cached:
        ax.scatter(xr, yr, s=12, alpha=0.45, c=[c], marker=mk,
                   edgecolors='white', linewidths=0.3, zorder=1)

    for st, xr, yr, c, mk in cached:
        fit = fit_linear_region(xr, yr, linear_region)
        if fit is None: 
            continue
        a, b, r2 = fit
        x_fit = np.linspace(global_x_min, global_x_max, 256) if extend_fit_to_full_span \
                else (xr if linear_region is None else xr[(xr>=linear_region[0]) & (xr<=linear_region[1])])
        y_fit = a * x_fit + b
        ax.plot(x_fit, y_fit, linewidth=1.2, color=c, zorder=2)

    ax.set_title(f"Zipf: model={model} | mode={mode} | elbow={elbow} | across steps")
    ax.set_xlabel("log(rank)"); ax.set_ylabel("log(value)")
    if ylim is not None: ax.set_ylim(*ylim)

    if show_colorbar:
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        cb = fig.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("step")

    ax.margins(x=0.02, y=0.05); fig.tight_layout()

    out_root = (OUT_DIR if OUT_DIR is not None else BASE_DIR / "plots_scheme_A_styled")
    out_dir  = ensure_dir(out_root / "by_model_across_steps")
    fname    = f"{safe_name(model)}_{mode}_elbow{elbow}_across_steps_seq.png"
    fpath    = out_dir / fname
    fig.savefig(fpath, dpi=dpi, bbox_inches="tight")
    plt.show()

    if skipped:
        print("Skipped steps:", skipped)
    return fpath

# =========================
# 3) Batch Exporters (PNG only)
# =========================
def batch_all_models_across_elbows_png(
    models_filter: Optional[List[str]] = None,       # None -> all models
    modes: Tuple[str, ...] = ("boost", "suppress"),
    steps_to_export: Tuple[Union[str,int], ...] = ("max",),
    linear_region: Optional[Tuple[float, float]] = (2.0, 3.3),
    ylim: Optional[Tuple[float, float]] = None,
    dpi: int = 300,
    cmap_name: str = "plasma",
    extend_fit_to_full_span: bool = True,
    show_colorbar: bool = True
) -> Dict[Tuple[str,str,Union[str,int]], Path]:
    """
    For each (model, mode, step), export one 'across elbows' PNG.
    """
    out: Dict[Tuple[str,str,Union[str,int]], Path] = {}
    all_models = sorted({e.model for e in entries})
    target_models = all_models if models_filter is None else [m for m in all_models if m in models_filter]

    for model in target_models:
        for mode in modes:
            if not any(e.model==model and e.mode==mode for e in entries):
                continue
            # avail steps (for warnings only)
            avail_steps = available_steps_for(model, mode)

            for st in steps_to_export:
                step_choice = st if (isinstance(st, str) and st=="max") else int(st)
                if step_choice != "max" and step_choice not in avail_steps:
                    print(f"[WARN] {model} | {mode}: step={step_choice} not found in some files; will skip those elbows.")
                png_path = plot_model_across_elbows(
                    model=model,
                    mode=mode,
                    step_choice=step_choice,
                    linear_region=linear_region,
                    ylim=ylim,
                    dpi=dpi,
                    cmap_name=cmap_name,
                    extend_fit_to_full_span=extend_fit_to_full_span,
                    show_colorbar=show_colorbar,
                    png_only=True
                )
                out[(model, mode, step_choice)] = png_path
    return out

def batch_fixed_elbow_across_steps_png(
    elbow: int,
    models_filter: Optional[List[str]] = None,       # None -> all models
    modes: Tuple[str, ...] = ("boost", "suppress"),
    steps_to_export: Optional[List[int]] = None,     # None -> use all steps in each CSV
    linear_region: Optional[Tuple[float, float]] = (2.0, 3.3),
    ylim: Optional[Tuple[float, float]] = None,
    dpi: int = 300,
    cmap_name: str = "plasma",
    extend_fit_to_full_span: bool = True,
    show_colorbar: bool = True
) -> Dict[Tuple[str,str,int], Path]:
    """
    For a fixed elbow, export one 'across steps' PNG per (model, mode).
    """
    out: Dict[Tuple[str,str,int], Path] = {}
    all_models = sorted({e.model for e in entries})
    target_models = all_models if models_filter is None else [m for m in all_models if m in models_filter]

    for model in target_models:
        for mode in modes:
            grp = [e for e in entries if e.model==model and e.mode==mode and e.elbow==elbow]
            if not grp:
                continue
            csv_path = grp[0].path
            try:
                avail = sorted(pd.read_csv(csv_path, usecols=["step"])["step"].unique().tolist())
            except Exception:
                avail = []

            steps_use = avail if (steps_to_export is None or len(steps_to_export)==0) \
                        else [int(s) for s in steps_to_export if int(s) in avail]
            missing = [] if steps_to_export is None else sorted(set(int(s) for s in steps_to_export) - set(steps_use))
            if missing:
                print(f"[WARN] {model} | {mode} | elbow={elbow} missing steps: {missing}")
            if len(steps_use) == 0:
                print(f"[INFO] {model} | {mode} | elbow={elbow}: no valid steps; skip.")
                continue

            png_path = plot_model_across_steps_at_elbow(
                model=model, mode=mode, elbow=elbow,
                steps_to_compare=steps_use,
                linear_region=linear_region, ylim=ylim,
                dpi=dpi, cmap_name=cmap_name,
                extend_fit_to_full_span=extend_fit_to_full_span,
                show_colorbar=show_colorbar,
                png_only=True
            )
            out[(model, mode, elbow)] = png_path
    return out

def batch_all_elbows_across_steps_png(
    elbows: Optional[List[int]] = None,               # None -> use discovered ELBOWS
    models_filter: Optional[List[str]] = None,
    modes: Tuple[str, ...] = ("boost", "suppress"),
    steps_to_export: Optional[List[int]] = None,      # None -> all steps per CSV
    linear_region: Optional[Tuple[float, float]] = (2.0, 3.3),
    ylim: Optional[Tuple[float, float]] = None,
    dpi: int = 300,
    cmap_name: str = "plasma",
    extend_fit_to_full_span: bool = True,
    show_colorbar: bool = True
) -> Dict[Tuple[int,str,str], Path]:
    """
    Run fixed-elbow-across-steps for EVERY elbow in `elbows` (or all discovered).
    Returns {(elbow, model, mode): png_path}
    """
    use_elbows = (ELBOWS if elbows is None else sorted(set(int(e) for e in elbows)))
    results: Dict[Tuple[int,str,str], Path] = {}
    total = 0
    print(f"Run ALL elbows: {use_elbows}")
    for el in use_elbows:
        per_elbow = batch_fixed_elbow_across_steps_png(
            elbow=el,
            models_filter=models_filter,
            modes=modes,
            steps_to_export=steps_to_export,
            linear_region=linear_region,
            ylim=ylim,
            dpi=dpi,
            cmap_name=cmap_name,
            extend_fit_to_full_span=extend_fit_to_full_span,
            show_colorbar=show_colorbar
        )
        for (model, mode, elbow_k), png_path in per_elbow.items():
            results[(elbow_k, model, mode)] = png_path
        total += len(per_elbow)
        print(f"[Elbow {el}] done: {len(per_elbow)} figure(s).")
    print(f"ALL elbows done. Total figures: {total}")
    return results


res_all = batch_all_elbows_across_steps_png(
    elbows=None,                          # None -> all discovered ELBOWS
    models_filter=None,
    modes=("boost","suppress"),
    steps_to_export=None,                 # None -> all steps per CSV
    linear_region=(4.5, 6.0),
    ylim=None,
    dpi=300,
    cmap_name="plasma"
)
print("Total PNGs:", len(res_all))
